In [1]:
'''script to regrid CMIP6 datatsets to target grid and store them'''
import numpy as np
import xarray as xr
import dask
import os
import intake
import pandas as pd
from collections import defaultdict
from tqdm.autonotebook import tqdm
from xmip.utils import google_cmip_col
from xmip.postprocessing import combine_datasets,_concat_sorted_time
from cmip_ds_dict_operations import select_period, drop_duplicate_timesteps, drop_coords, drop_incomplete
import xesmf as xe
import gcsfs
from get_era5_around_tgs import get_era5_around_tgs
fs = gcsfs.GCSFileSystem() #list stores, stripp zarr from filename, load 

/tmp/ipykernel_758/3429241155.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
#configure settings
output_path = 'gs://leap-persistent/timh37/HighResMIP/surgeNN_predictors/era5_grid/'
overwrite_existing = False #whether or not to process files for which output already exists in the output path

query_vars = ['psl','uas','vas'] #variables to process
required_vars = ['psl','uas','vas'] #variables that includes models should provide
freqs = ['6hrPlevPt']#['3hr','E3hr']

highresmip_model = 'CMCC-CM2-VHR4'#'HadGEM3-GC31-HM'
experiments= ['highres-future','hist-1950']

In [3]:
#query simulations & manipulate data catalog:
col = intake.open_esm_datastore("https://storage.googleapis.com/cmip6/cmip6-pgf-ingestion-test/catalog/catalog.json") #temporary pangeo-leap-forge catalogue
cat = col.search(activity_id='HighResMIP',table_id=freqs,source_id=[highresmip_model],experiment_id=experiments,variable_id=query_vars)#table_id='3hr',require_all_on=['member_id','grid_label','experiment_id'])
kwargs = {'zarr_kwargs':{'consolidated':True,'use_cftime':True},'aggregate':False} #keyword arguments for generating dictionary of datasets from cmip6 catalogue
ddict = cat.to_dataset_dict(**kwargs) #open datasets into dictionary
ddict = drop_duplicate_timesteps(ddict) #remove duplicate timesteps from ds if present
ddict = drop_coords(ddict,['bnds','nbnd','height']) #remove some unused auxiliary coordinates
with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    hist_fut = combine_datasets(ddict,_concat_sorted_time,match_attrs =['source_id', 'grid_label','table_id','variant_label','variable_id'],combine_func_kwargs={'join':'inner','coords':'minimal'})    
hist_fut = drop_duplicate_timesteps(hist_fut) 
hist_fut = drop_incomplete(hist_fut)

/srv/conda/envs/notebook/lib/python3.12/site-packages/intake_esm/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution



--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.member_id.table_id.variable_id.grid_label.sub_experiment_id.variant_label.version.zstore'


In [4]:
cat = col.search(activity_id='HighResMIP',table_id=freqs,source_id=[highresmip_model],experiment_id=experiments)
cat.df

,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,sub_experiment_id,variant_label,version,zstore
0,HighResMIP,CMCC,CMCC-CM2-VHR4,highres-future,r1i1p1f1,6hrPlevPt,psl,gn,none,r1i1p1f1,v20190509,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...
1,HighResMIP,CMCC,CMCC-CM2-VHR4,hist-1950,r1i1p1f1,6hrPlevPt,psl,gn,none,r1i1p1f1,v20180705,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...
2,HighResMIP,CMCC,CMCC-CM2-VHR4,highres-future,r1i1p1f1,6hrPlevPt,uas,gn,none,r1i1p1f1,v20190509,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...
3,HighResMIP,CMCC,CMCC-CM2-VHR4,highres-future,r1i1p1f1,6hrPlevPt,vas,gn,none,r1i1p1f1,v20190509,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...
4,HighResMIP,CMCC,CMCC-CM2-VHR4,hist-1950,r1i1p1f1,6hrPlevPt,uas,gn,none,r1i1p1f1,v20180705,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...
5,HighResMIP,CMCC,CMCC-CM2-VHR4,hist-1950,r1i1p1f1,6hrPlevPt,vas,gn,none,r1i1p1f1,v20180705,gs://cmip6/cmip6-pgf-ingestion-test/zarr_store...


In [5]:
#regrid to ERA5 (does this make sense if training only on highresmip?)
target_grid = xr.Dataset( #grid to interpolate CMIP6 simulations to
        {   "longitude": (["longitude"], np.arange(-13,13,.25), {"units": "degrees_east"}),
            "latitude": (["latitude"], np.arange(63,34,-.25), {"units": "degrees_north"}),})

regridded_datasets = defaultdict(dict)
for key,ds in tqdm(hist_fut.items()):
    ds.coords['lon'] = (ds.coords['lon'] + 180) % 360 - 180
    ds = ds.sortby(ds.lon)

    ds = ds.where((ds.lat>30)&(ds.lat<65)&(ds.lon>-15)&(ds.lon<15),drop=True) #reduce to Europe (not sure if necessary/more efficient)
    ds = ds.isel(sub_experiment_id=0,drop=True) #remove this coordinate
    
    regridder = xe.Regridder(ds,target_grid,'bilinear',ignore_degenerate=True,periodic=True)
    regridded_ds = regridder(ds.chunk({'time':2000,'lat':10000,'lon':10000}),keep_attrs=True)

    regridded_datasets[key] = regridded_ds.unify_chunks().chunk({'time':500})

  0%|          | 0/3 [00:00<?, ?it/s]

In [6]:
predictors_eu = xr.merge(list(regridded_datasets.values()))
predictors_eu  = predictors_eu.rename({'vas':'v10','uas':'u10','psl':'msl'})
predictors_eu.attrs['resolution'] = '0p25x0p25'

/tmp/ipykernel_758/3755880315.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  predictors_eu = xr.merge(list(regridded_datasets.values()))


In [7]:
predictors_eu = predictors_eu.load() #this takes a while (~40GB, depending on target grid)

In [8]:
#get predictors around each tide gauge and store
grid_size_around_tgs=5 #degrees around TGs
tg_coords = xr.open_dataset('../../input/gesla3_tg_coordinates_eu.nc')#.sel(tg=['den_helder-denhdr-nld-rws.csv']) #load TG coordinates

tgs        = ['stavanger-svg-nor-nhs.csv','wick-wic-gbr-bodc.csv','esbjerg-esb-dnk-dmi.csv',
                  'immingham-imm-gbr-bodc.csv','den_helder-denhdr-nld-rws.csv', 'fishguard-fis-gbr-bodc.csv',  
                  'brest-822a-fra-uhslc.csv', 'vigo-vigo-esp-ieo.csv',  'alicante_i_outer_harbour-alio-esp-da_mm.csv']

for output_tg in tqdm(tgs):#tg_coords.tg.values:
    print('processing: '+output_tg)
    predictors = get_era5_around_tgs(predictors_eu,grid_size_around_tgs,tg_coords.sel(tg=[output_tg])).chunk({'tg':1,'time':1000000})
    predictors = predictors.isel(variant_label=0,tg=0)
    predictors['tg'] = predictors['tg'].values.astype(np.dtypes.StringDType)
    output_fn = os.path.join(output_path,highresmip_model,'predictors_'+predictors.source_id+'_'+str(predictors.time.dt.year[0].values)+'_'+str(predictors.time.dt.year[-1].values)+'_'+output_tg.replace('.csv','')+'.zarr')
    predictors.drop_encoding().to_zarr(output_fn,mode='w') #store 

  0%|          | 0/9 [00:00<?, ?it/s]

processing: stavanger-svg-nor-nhs.csv


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


processing: wick-wic-gbr-bodc.csv


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


...

/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


processing: vigo-vigo-esp-ieo.csv


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


processing: alicante_i_outer_harbour-alio-esp-da_mm.csv


/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/core/dtype/npy/string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=8, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/api/asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
